<a href="https://colab.research.google.com/github/sumaiiiyyah/code/blob/master/ALDH1A1_Top1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ALDH1A1 (PDB: 4X4L) — Top-10 to Top-1 Candidate Evaluation

**Workflow template:** PAPSS2 Top-1 New Derivatives notebook, adapted to the supplied ALDH1A1 Top-10 workbook and exact SDF.

**Purpose:** audit the authoritative docking results and ligand structures; calculate RDKit descriptors; run ADMET-AI; retrieve exact-compound and scaffold literature; integrate evidence; prepare the selected-candidate decision records, reports, methodology, and one verified ZIP archive.

**Important interpretation rules**
- The workbook's `Overall Rank` is preserved. Edock is independently ranked ascending (more negative = better) as a separate comparison.
- The supplied SDF is the structure source for descriptor and ADMET calculations. CID matching is checked explicitly.
- The workbook itself warns that the PubChem CID structure must be verified against the exact docked derivative. A matching CID alone does not prove exact chemical identity; structure comparisons and warnings are retained.
- ADMET-AI predictions are computational model outputs, not experimental measurements or clinical probabilities. A classifier value ≥0.5 is an exploratory screening flag only.
- DrugBank-approved percentiles, when returned by ADMET-AI, are relative reference-distribution values, not probabilities.
- No arbitrary composite ADMET score is created. Final candidate assessment is evidence-based and transparent.
- Literature keyword matches are screening leads, not confirmation of exact-compound activity. Manual verification fields are included.
- Docking scores are not binding affinities; no experimental validation is implied.

**Run cells in order.** The final cell builds and validates one ZIP file for download. External API/model stages are protected so that a temporary service failure is logged rather than silently treated as a negative result.


## 0. Upload the two ALDH1A1 source files

In [1]:
from google.colab import files
from pathlib import Path
import shutil, os, re, json, zipfile, hashlib, warnings, subprocess, sys, importlib.util
import pandas as pd
import numpy as np

print("Upload BOTH files:")
print("1. ALDH1A1_4X4L_Overall_Top10_NEW_docked.xlsx")
print("2. ALDH1A1_4X4L_Overall_Top10_Exact.sdf")
uploaded = files.upload()

xlsx_candidates = [Path("/content") / n for n in uploaded if n.lower().endswith((".xlsx", ".xls"))]
sdf_candidates = [Path("/content") / n for n in uploaded if n.lower().endswith(".sdf")]

# Also accept already-present files in /content (e.g., reruns).
if not xlsx_candidates:
    xlsx_candidates = list(Path("/content").glob("*ALDH1A1*Top10*.xlsx"))
if not sdf_candidates:
    sdf_candidates = list(Path("/content").glob("*ALDH1A1*Top10*.sdf"))

if not xlsx_candidates:
    raise FileNotFoundError("ALDH1A1 Top-10 Excel workbook was not uploaded.")
if not sdf_candidates:
    raise FileNotFoundError("ALDH1A1 exact Top-10 SDF was not uploaded.")

# Prefer the expected source names if more than one candidate is present.
XLSX = next((p for p in xlsx_candidates if "NEW_docked" in p.name), xlsx_candidates[0])
SDF = next((p for p in sdf_candidates if "Exact" in p.name), sdf_candidates[0])

BASE = Path("/content/ALDH1A1_Top10_to_Top1")
FINAL = Path("/content/ALDH1A1_Top10_to_Top1_FINAL")
ZIP_PATH = Path("/content/ALDH1A1_Top10_to_Top1_FINAL.zip")

FOLDER_NAMES = [
    "01_Source_Files",
    "02_Data_Quality_and_Structure_Audit",
    "03_Docking_Results",
    "04_Molecular_Properties_and_ADMET",
    "05_Literature_Analysis",
    "06_Integrated_Candidate_Assessment",
    "07_Final_Reports",
    "08_Methodology",
    "09_Selected_Candidate_Structure"
]

# Rebuild working directory cleanly; source files are copied unchanged.
if BASE.exists():
    shutil.rmtree(BASE)
for name in FOLDER_NAMES:
    (BASE / name).mkdir(parents=True, exist_ok=True)

shutil.copy2(XLSX, BASE / "01_Source_Files" / "01_Authoritative_ALDH1A1_Top10_Docking_Input.xlsx")
shutil.copy2(SDF, BASE / "01_Source_Files" / "02_ALDH1A1_Top10_Exact_Ligand_Structures.sdf")

print("\\nINPUTS ACCEPTED")
print("Workbook:", XLSX.name, "|", XLSX.stat().st_size, "bytes")
print("SDF:", SDF.name, "|", SDF.stat().st_size, "bytes")
print("Working directory:", BASE)


Upload BOTH files:
1. ALDH1A1_4X4L_Overall_Top10_NEW_docked.xlsx
2. ALDH1A1_4X4L_Overall_Top10_Exact.sdf


Saving ALDH1A1_4X4L_Overall_Top10_Exact.sdf to ALDH1A1_4X4L_Overall_Top10_Exact.sdf
Saving ALDH1A1_4X4L_Overall_Top10_NEW_docked.xlsx to ALDH1A1_4X4L_Overall_Top10_NEW_docked.xlsx
\nINPUTS ACCEPTED
Workbook: ALDH1A1_4X4L_Overall_Top10_NEW_docked.xlsx | 33435 bytes
SDF: ALDH1A1_4X4L_Overall_Top10_Exact.sdf | 76733 bytes
Working directory: /content/ALDH1A1_Top10_to_Top1


## 1. Install and import core dependencies

In [2]:
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("rdkit")
ensure_package("openpyxl")
ensure_package("docx", "python-docx")
ensure_package("requests")

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED, rdMolDescriptors
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from docx import Document
from docx.shared import Pt, Inches
import requests, time, traceback

print("Core imports successful.")


Core imports successful.


## 2. Authoritative docking workbook audit

In [3]:
raw = pd.read_excel(XLSX)
print("Workbook shape:", raw.shape)
print("Sheet:", pd.ExcelFile(XLSX).sheet_names)
print("Columns:", raw.columns.tolist())

required = ["PubChem CID", "Edock", "Overall Rank", "Library", "Original Edock", "Redocking Edock"]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Required workbook columns missing: {missing}")

def clean_cid(x):
    if pd.isna(x): return ""
    return re.sub(r"\.0$", "", str(x).strip())

raw["PubChem CID"] = raw["PubChem CID"].apply(clean_cid)
raw["Edock"] = pd.to_numeric(raw["Edock"], errors="coerce")
raw["Original Edock"] = pd.to_numeric(raw["Original Edock"], errors="coerce")
raw["Redocking Edock"] = pd.to_numeric(raw["Redocking Edock"], errors="coerce")
raw["Overall Rank"] = pd.to_numeric(raw["Overall Rank"], errors="coerce")

if len(raw) != 10:
    raise ValueError(f"Expected exactly 10 candidates; found {len(raw)}.")
if raw["PubChem CID"].eq("").any() or raw["PubChem CID"].nunique() != 10:
    raise ValueError("Expected 10 unique, non-missing PubChem CIDs.")
if raw["Edock"].isna().any() or raw["Overall Rank"].isna().any():
    raise ValueError("Missing Edock or Overall Rank values.")

raw["Edock_Rank_Recalculated"] = raw["Edock"].rank(method="min", ascending=True).astype(int)
raw["Edock_vs_Original_Delta"] = raw["Edock"] - raw["Original Edock"]
raw["Redocking_vs_Original_Delta"] = raw["Redocking Edock"] - raw["Original Edock"]
raw["Redocking_vs_Current_Edock_Delta"] = raw["Redocking Edock"] - raw["Edock"]

rank_comparison = raw[[
    "Overall Rank", "Edock_Rank_Recalculated", "Library", "PubChem CID",
    "Original Edock", "Edock", "Redocking Edock",
    "Edock_vs_Original_Delta", "Redocking_vs_Original_Delta",
    "Redocking_vs_Current_Edock_Delta", "Structure source", "Structure warning"
]].sort_values("Overall Rank")

audit_path = BASE / "02_Data_Quality_and_Structure_Audit" / "01_ALDH1A1_Compound_Data_Audit.xlsx"
with pd.ExcelWriter(audit_path, engine="openpyxl") as w:
    rank_comparison.to_excel(w, sheet_name="Authoritative_Top10", index=False)
    pd.DataFrame({"Column": raw.columns, "Missing_Count": raw.isna().sum().values}).to_excel(w, sheet_name="Missing_Value_Audit", index=False)

print("Candidate count:", len(raw))
print("Unique CIDs:", raw["PubChem CID"].nunique())
print("\\nOriginal workbook rank vs independently recalculated Edock rank:")
display(rank_comparison)
print("\\nSaved:", audit_path)


Workbook shape: (10, 68)
Sheet: ['Sheet0']
Columns: ['mol', 'L', 'IX', 'NAME', 'Edock', 'Egb', 'Ege', 'Egs', 'Egv', 'CONF', 'RecConf', 'Molecule Name', 'PUBCHEM COMPOUND CID', 'PUBCHEM COMPOUND CANONICALIZED', 'PUBCHEM CONFORMER RMSD', 'PUBCHEM CACTVS COMPLEXITY', 'PUBCHEM CONFORMER DIVERSEORDER', 'PUBCHEM CACTVS HBOND ACCEPTOR', 'PUBCHEM MMFF94 PARTIAL CHARGES', 'PUBCHEM CACTVS HBOND DONOR', 'PUBCHEM EFFECTIVE ROTOR COUNT', 'PUBCHEM CACTVS ROTATABLE BOND', 'PUBCHEM PHARMACOPHORE FEATURES', 'PUBCHEM CACTVS SUBSKEYS', 'PUBCHEM IUPAC OPENEYE NAME', 'PUBCHEM IUPAC CAS NAME', 'PUBCHEM IUPAC NAME MARKUP', 'PUBCHEM IUPAC NAME', 'PUBCHEM IUPAC SYSTEMATIC NAME', 'PUBCHEM IUPAC TRADITIONAL NAME', 'PUBCHEM IUPAC INCHI', 'PUBCHEM IUPAC INCHIKEY', 'PUBCHEM CONFORMER ID', 'PUBCHEM XLOGP3 AA', 'PUBCHEM MMFF94 ENERGY', 'PUBCHEM XLOGP3', 'PUBCHEM EXACT MASS', 'PUBCHEM FEATURE SELFOVERLAP', 'PUBCHEM MOLECULAR FORMULA', 'PUBCHEM SHAPE FINGERPRINT', 'PUBCHEM SHAPE MULTIPOLES', 'PUBCHEM MOLECULAR WEIGHT',

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Overall Rank,Edock_Rank_Recalculated,Library,PubChem CID,Original Edock,Edock,Redocking Edock,Edock_vs_Original_Delta,Redocking_vs_Original_Delta,Redocking_vs_Current_Edock_Delta,Structure source,Structure warning
0,1,10,Thiomorpholine,173315143,-27.35858,-27.35929,-196.82480,-0.00071,-169.46622,-169.46551,PubChem CID lookup,Verify against the exact docked derivative bef...
1,2,1,Thiophene,153531369,-198.52530,-200.09910,-153.07170,-1.57380,45.45360,47.02740,PubChem CID lookup,Verify against the exact docked derivative bef...
2,3,3,Thiophene,150310321,-128.30920,-130.70490,-128.87660,-2.39570,-0.56740,1.82830,PubChem CID lookup,Verify against the exact docked derivative bef...
3,4,2,Thiophene,153531370,-152.20340,-153.06790,-127.23660,-0.86450,24.96680,25.83130,PubChem CID lookup,Verify against the exact docked derivative bef...
4,5,4,Thiophene,175018237,-127.54010,-129.09780,-104.50900,-1.55770,23.03110,24.58880,PubChem CID lookup,Verify against the exact docked derivative bef...
5,6,5,Indole,12592795,-118.83710,-104.91610,-104.16600,13.92100,14.67110,0.75010,PubChem CID lookup,Verify against the exact docked derivative bef...
6,7,6,Indole,12592794,-91.99455,-88.92295,-89.92871,3.07160,2.06584,-1.00576,PubChem CID lookup,Verify against the exact docked derivative bef...
7,8,8,Benzoxazole,21260039,-85.10348,-86.31135,-88.85378,-1.20787,-3.75030,-2.54243,PubChem CID lookup,Verify against the exact docked derivative bef...
8,9,7,Benzimidazole,58965179,-89.69313,-88.55415,-88.29002,1.13898,1.40311,0.26413,PubChem CID lookup,Verify against the exact docked derivative bef...
9,10,9,Benzimidazole,14157051,-77.19590,-78.31743,-78.32375,-1.12153,-1.12785,-0.00632,PubChem CID lookup,Verify against the exact docked derivative bef...


\nSaved: /content/ALDH1A1_Top10_to_Top1/02_Data_Quality_and_Structure_Audit/01_ALDH1A1_Compound_Data_Audit.xlsx


## 3. Validate SDF records and match structures to CIDs

In [4]:
supplier = Chem.SDMolSupplier(str(SDF), removeHs=False, sanitize=True)
sdf_records, invalid_records = [], []

for idx, mol in enumerate(supplier, start=1):
    if mol is None:
        invalid_records.append(idx)
        continue
    props = {k: mol.GetProp(k) for k in mol.GetPropNames(includePrivate=True, includeComputed=False)}
    cid = clean_cid(props.get("PUBCHEM_COMPOUND_CID", props.get("PubChem_CID", props.get("PubChem CID", ""))))
    clean_mol = Chem.RemoveHs(mol)
    sdf_records.append({
        "SDF_Record": idx,
        "SDF_Name": mol.GetProp("_Name") if mol.HasProp("_Name") else "",
        "SDF_CID": cid,
        "SDF_Formula": rdMolDescriptors.CalcMolFormula(clean_mol),
        "SDF_Molecular_Weight": Descriptors.MolWt(clean_mol),
        "SDF_Canonical_Isomeric_SMILES": Chem.MolToSmiles(clean_mol, canonical=True, isomericSmiles=True),
        "SDF_InChIKey": Chem.MolToInchiKey(clean_mol),
        "Heavy_Atoms": clean_mol.GetNumHeavyAtoms(),
        "Isotope_Atom_Count": sum(a.GetIsotope() != 0 for a in clean_mol.GetAtoms()),
        "SDF_Properties": json.dumps(props, default=str)
    })

sdf_df = pd.DataFrame(sdf_records)
if len(sdf_df) != 10:
    raise ValueError(f"Expected 10 valid SDF records; found {len(sdf_df)}. Invalid records: {invalid_records}")
if sdf_df["SDF_CID"].eq("").any() or sdf_df["SDF_CID"].nunique() != 10:
    raise ValueError("SDF must contain 10 unique PubChem CIDs.")

match = raw.merge(sdf_df, left_on="PubChem CID", right_on="SDF_CID", how="outer", indicator=True, validate="one_to_one")
match["CID_Match"] = match["_merge"].eq("both")
match["Structure_Identity_Flag"] = np.where(
    match["CID_Match"],
    "CID matched; exact docked-derivative identity still requires source-structure verification",
    "CID mismatch — HOLD"
)

# Compare available workbook formula/SMILES fields with SDF.
formula_col = next((c for c in raw.columns if c.upper() in ["PUBCHEM MOLECULAR FORMULA", "MOLECULAR FORMULA"]), None)
smiles_col = next((c for c in raw.columns if c.upper() in ["PUBCHEM SMILES", "SMILES", "PUBCHEM CONNECTIVITY SMILES"]), None)
if formula_col:
    match["Workbook_Formula"] = match[formula_col]
    match["Formula_Match"] = match.apply(lambda r: str(r.get(formula_col, "")).strip() == str(r.get("SDF_Formula", "")).strip(), axis=1)
if smiles_col:
    match["Workbook_SMILES"] = match[smiles_col]
    match["Workbook_SMILES_Present"] = match[smiles_col].notna() & match[smiles_col].astype(str).str.strip().ne("")

match_path = BASE / "02_Data_Quality_and_Structure_Audit" / "02_ALDH1A1_SDF_Structure_and_CID_Matching.xlsx"
with pd.ExcelWriter(match_path, engine="openpyxl") as w:
    sdf_df.to_excel(w, sheet_name="SDF_Validation", index=False)
    match.to_excel(w, sheet_name="CID_Matching", index=False)

print("SDF valid records:", len(sdf_df))
print("Invalid records:", invalid_records)
print("CID matches:", int(match["CID_Match"].sum()))
print("Formula column found:", formula_col)
print("SMILES column found:", smiles_col)
display(match[["Overall Rank", "PubChem CID", "SDF_Record", "SDF_Name", "SDF_Formula", "SDF_Molecular_Weight", "CID_Match", "Structure_Identity_Flag"]].sort_values("Overall Rank"))
print("Saved:", match_path)


SDF valid records: 10
Invalid records: []
CID matches: 10
Formula column found: PUBCHEM MOLECULAR FORMULA
SMILES column found: PUBCHEM SMILES


,Overall Rank,PubChem CID,SDF_Record,SDF_Name,SDF_Formula,SDF_Molecular_Weight,CID_Match,Structure_Identity_Flag
6,1,173315143,1,Thiomorpholine_Rank10_CID173315143,C4H9FNS+,122.188,True,CID matched; exact docked-derivative identity ...
4,2,153531369,2,Thiophene_Rank1_CID153531369,C4H4S17,597.215,True,CID matched; exact docked-derivative identity ...
3,3,150310321,3,Thiophene_Rank3_CID150310321,C4H4S11,404.813,True,CID matched; exact docked-derivative identity ...
5,4,153531370,4,Thiophene_Rank2_CID153531370,C4H4S13,468.947,True,CID matched; exact docked-derivative identity ...
7,5,175018237,5,Thiophene_Rank4_CID175018237,C4H4S11,404.813,True,CID matched; exact docked-derivative identity ...
1,6,12592795,6,Indole_Rank1_CID12592795,C62H98ClNO8,1020.918,True,CID matched; exact docked-derivative identity ...
0,7,12592794,7,Indole_Rank2_CID12592794,C54H82ClNO8,908.702,True,CID matched; exact docked-derivative identity ...
8,8,21260039,8,Benzoxazole_Rank1_CID21260039,C29H50NO+,428.725,True,CID matched; exact docked-derivative identity ...
9,9,58965179,9,Benzimidazole_Rank1_CID58965179,C22H39N5,373.589,True,CID matched; exact docked-derivative identity ...
2,10,14157051,10,Benzimidazole_Rank2_CID14157051,C23H39N2+,343.579,True,CID matched; exact docked-derivative identity ...


Saved: /content/ALDH1A1_Top10_to_Top1/02_Data_Quality_and_Structure_Audit/02_ALDH1A1_SDF_Structure_and_CID_Matching.xlsx


## 4. RDKit molecular properties and structural alerts

In [5]:
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
pains_catalog = FilterCatalog(params)

desc_rows = []
for _, row in sdf_df.iterrows():
    mol = Chem.MolFromSmiles(row["SDF_Canonical_Isomeric_SMILES"])
    if mol is None:
        raise ValueError(f"RDKit could not reconstruct CID {row['SDF_CID']}")
    alerts = pains_catalog.GetMatches(mol)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd, hba = Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol)
    desc_rows.append({
        "PubChem CID": row["SDF_CID"],
        "SDF_Record": row["SDF_Record"],
        "Molecular_Weight_RDKit": mw,
        "Exact_Molecular_Weight": Descriptors.ExactMolWt(mol),
        "cLogP_RDKit": logp,
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": hbd,
        "HBA": hba,
        "Rotatable_Bonds": Lipinski.NumRotatableBonds(mol),
        "Ring_Count": rdMolDescriptors.CalcNumRings(mol),
        "Aromatic_Ring_Count": rdMolDescriptors.CalcNumAromaticRings(mol),
        "Fraction_CSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "Heavy_Atom_Count": mol.GetNumHeavyAtoms(),
        "Formal_Charge": Chem.GetFormalCharge(mol),
        "QED": QED.qed(mol),
        "Lipinski_Violations": int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10),
        "PAINS_Alert": "; ".join(sorted(set(x.GetDescription() for x in alerts))) if alerts else "None detected",
        "SDF_Isotope_Atoms": row["Isotope_Atom_Count"]
    })

descriptors = pd.DataFrame(desc_rows)
flag_cols = ["High_Molecular_Weight", "High_Lipophilicity", "Very_Low_TPSA", "Low_QED", "Lipinski_Violation_Flag", "PAINS_Flag"]
descriptors["High_Molecular_Weight"] = descriptors["Molecular_Weight_RDKit"] > 500
descriptors["High_Lipophilicity"] = descriptors["cLogP_RDKit"] > 5
descriptors["Very_Low_TPSA"] = descriptors["TPSA"] < 20
descriptors["Low_QED"] = descriptors["QED"] < 0.30
descriptors["Lipinski_Violation_Flag"] = descriptors["Lipinski_Violations"] > 0
descriptors["PAINS_Flag"] = ~descriptors["PAINS_Alert"].eq("None detected")
descriptors["Total_Property_Flags"] = descriptors[flag_cols].sum(axis=1)

properties = raw.merge(descriptors, on="PubChem CID", how="left", validate="one_to_one")
prop_path = BASE / "04_Molecular_Properties_and_ADMET" / "03_ALDH1A1_RDKit_Molecular_Properties.xlsx"
with pd.ExcelWriter(prop_path, engine="openpyxl") as w:
    properties.to_excel(w, sheet_name="Molecular_Properties", index=False)
    descriptors.to_excel(w, sheet_name="RDKit_Descriptors", index=False)

display(properties[["Overall Rank", "PubChem CID", "Molecular_Weight_RDKit", "cLogP_RDKit", "TPSA", "HBD", "HBA", "QED", "Lipinski_Violations", "PAINS_Alert", "Total_Property_Flags"]].sort_values("Overall Rank"))
print("Saved:", prop_path)


,Overall Rank,PubChem CID,Molecular_Weight_RDKit,cLogP_RDKit,TPSA,HBD,HBA,QED,Lipinski_Violations,PAINS_Alert,Total_Property_Flags
0,1,173315143,122.188,-0.49740,4.44,1,1,0.424462,0,None detected,1
1,2,153531369,597.215,11.75980,0.00,1,17,0.115672,3,None detected,5
2,3,150310321,404.813,7.87060,0.00,1,11,0.245544,2,None detected,4
3,4,153531370,468.947,9.16700,0.00,1,13,0.187433,2,None detected,4
4,5,175018237,404.813,7.87060,0.00,1,11,0.245544,2,None detected,4
5,6,12592795,1020.918,17.92432,110.13,0,8,0.031272,2,None detected,4
6,7,12592794,908.702,14.80352,110.13,0,8,0.031952,2,None detected,4
7,8,21260039,428.725,9.54220,17.02,0,1,0.142789,1,None detected,4
8,9,58965179,373.589,3.75420,78.76,4,4,0.318452,0,None detected,0
9,10,14157051,343.579,6.93670,19.67,1,0,0.266886,1,None detected,4


Saved: /content/ALDH1A1_Top10_to_Top1/04_Molecular_Properties_and_ADMET/03_ALDH1A1_RDKit_Molecular_Properties.xlsx


## 5. ADMET-AI predictions (all 10 compounds)

In [6]:
admet_status = "NOT_RUN"
admet_error = ""
admet_results = pd.DataFrame()
admet_metadata = pd.DataFrame()

try:
    ensure_package("admet_ai", "admet-ai")
    import admet_ai
    from admet_ai import ADMETModel

    smiles_df = sdf_df[["SDF_CID", "SDF_Canonical_Isomeric_SMILES"]].rename(
        columns={"SDF_CID": "PubChem CID", "SDF_Canonical_Isomeric_SMILES": "SMILES"}
    )
    print("Running ADMET-AI on 10 SDF-derived structures...")
    model = ADMETModel()
    pred = model.predict(smiles=smiles_df["SMILES"].tolist()).reset_index(drop=True)

    if len(pred) != 10:
        raise ValueError(f"ADMET-AI returned {len(pred)} rows, expected 10.")

    admet_results = pd.concat([smiles_df.reset_index(drop=True), pred.reset_index(drop=True)], axis=1)
    admet_results = raw[["PubChem CID", "Overall Rank", "Library", "Edock"]].merge(
        admet_results, on="PubChem CID", how="left", validate="one_to_one"
    )
    if admet_results.drop(columns=["PubChem CID", "Overall Rank", "Library", "Edock", "SMILES"], errors="ignore").isna().all().all():
        raise ValueError("ADMET-AI returned no usable endpoint values.")

    toxicity_endpoints = [
        "AMES", "Carcinogens_Lagunin", "ClinTox", "DILI", "hERG",
        "Skin_Reaction", "NR-AR", "NR-AR-LBD", "NR-AhR",
        "NR-Aromatase", "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
        "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53", "LD50_Zhu"
    ]
    available_tox = [c for c in toxicity_endpoints if c in admet_results.columns]
    classifier_cols = [c for c in ["AMES", "Carcinogens_Lagunin", "ClinTox", "DILI", "Skin_Reaction", "hERG"] if c in admet_results.columns]

    if classifier_cols:
        admet_results["Classifier_Signals_At_Or_Above_0.5"] = (
            admet_results[classifier_cols].apply(pd.to_numeric, errors="coerce") >= 0.5
        ).sum(axis=1)

    percentile_cols = [c for c in admet_results.columns if c.endswith("_drugbank_approved_percentile")]
    admet_status = "SUCCESS"

    admet_path = BASE / "04_Molecular_Properties_and_ADMET" / "04_ALDH1A1_ADMET_AI_All10.xlsx"
    with pd.ExcelWriter(admet_path, engine="openpyxl") as w:
        admet_results.to_excel(w, sheet_name="All_Raw_Predictions", index=False)
        admet_results[["PubChem CID"] + available_tox].to_excel(w, sheet_name="Toxicity_Predictions", index=False)
        admet_results[["PubChem CID"] + [c for c in admet_results.columns if c not in available_tox and c != "PubChem CID"]].to_excel(w, sheet_name="Other_ADMET", index=False)
        pd.DataFrame({"Endpoint": admet_results.columns}).to_excel(w, sheet_name="Endpoint_Inventory", index=False)
    print("ADMET-AI version:", getattr(admet_ai, "__version__", "version unavailable"))
    print("Prediction endpoints:", len([c for c in admet_results.columns if c not in ["PubChem CID", "Overall Rank", "Library", "Edock", "SMILES"]]))
    print("Toxicity endpoints found:", len(available_tox))
    print("DrugBank percentile endpoints found:", len(percentile_cols))
    print("Saved:", admet_path)

except Exception as e:
    admet_status = "FAILED_OR_UNAVAILABLE"
    admet_error = repr(e)
    print("ADMET-AI stage could not complete:", admet_error)
    print("The pipeline will continue and record ADMET as unavailable; no missing prediction is interpreted as a favourable result.")


Running ADMET-AI on 10 SDF-derived structures...


model ensembles:   0%|          | 0/2 [00:00<?, ?it/s]INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

Output()

... (6 outputs left out)

Output()

Output()

model ensembles: 100%|██████████| 2/2 [00:01<00:00,  1.58it/s]


ADMET-AI version: 2.0.1
Prediction endpoints: 105
Toxicity endpoints found: 19
DrugBank percentile endpoints found: 52
Saved: /content/ALDH1A1_Top10_to_Top1/04_Molecular_Properties_and_ADMET/04_ALDH1A1_ADMET_AI_All10.xlsx


## 6. Literature retrieval — exact compound and scaffold searches

In [7]:
# Search OpenAlex and Crossref using candidate names, CIDs and structural families.
# Retrieval is exploratory. Results require manual relevance and exact-identity review.

session = requests.Session()
session.headers.update({"User-Agent": "ALDH1A1-computational-literature-audit/1.0 (academic research)"})

def first_name(row):
    for c in ["PUBCHEM IUPAC NAME", "Molecule Name", "NAME", "PUBCHEM IUPAC SYSTEMATIC NAME"]:
        v = row.get(c)
        if pd.notna(v) and str(v).strip() and str(v).lower() != "nan":
            return str(v).strip()
    return ""

scaffold_map = {
    "Thiomorpholine": "thiomorpholine",
    "Thiophene": "thiophene",
    "Indole": "indole",
    "Benzoxazole": "benzoxazole",
    "Benzimidazole": "benzimidazole"
}
tasks = []
for _, r in raw.iterrows():
    cid = r["PubChem CID"]
    name = first_name(r)
    scaffold = scaffold_map.get(str(r["Library"]), str(r["Library"]).lower())
    tasks.append({"CID": cid, "Compound": name, "Evidence_Type": "Exact compound", "Query": f'"{cid}"'})
    if name:
        tasks.append({"CID": cid, "Compound": name, "Evidence_Type": "Exact compound name", "Query": f'"{name}"'})
    tasks.append({"CID": cid, "Compound": name, "Evidence_Type": "Scaffold anticancer", "Query": f'{scaffold} AND (anticancer OR cytotoxicity OR cancer)'})
    tasks.append({"CID": cid, "Compound": name, "Evidence_Type": "Scaffold breast cancer", "Query": f'{scaffold} AND ("breast cancer" OR MCF-7 OR MDA-MB-231)'})

lit_records, search_log = [], []
for n, task in enumerate(tasks, start=1):
    success, error, count = False, "", 0
    try:
        resp = session.get("https://api.openalex.org/works", params={"search": task["Query"], "per-page": 10}, timeout=35)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        success, count = True, len(results)
        for item in results:
            doi = item.get("doi") or ""
            title = item.get("title") or ""
            year = item.get("publication_year")
            journal = ((item.get("primary_location") or {}).get("source") or {}).get("display_name") or ""
            authors = "; ".join([a.get("author", {}).get("display_name", "") for a in (item.get("authorships") or [])[:8]])
            abstract = ""
            inv = item.get("abstract_inverted_index") or {}
            if inv:
                words = {}
                for word, positions in inv.items():
                    for pos in positions: words[pos] = word
                abstract = " ".join(words[k] for k in sorted(words))
            lit_records.append({
                **task, "Title": title, "Year": year, "DOI": doi,
                "Journal": journal, "Authors": authors, "Abstract": abstract,
                "URL": item.get("id", ""), "Source_Database": "OpenAlex"
            })
    except Exception as e:
        error = repr(e)
    search_log.append({**task, "Successful": success, "Results_Returned": count, "Error": error})
    time.sleep(0.15)

literature = pd.DataFrame(lit_records)
log_df = pd.DataFrame(search_log)

if not literature.empty:
    literature["DOI_Normalised"] = literature["DOI"].fillna("").astype(str).str.lower().str.replace("https://doi.org/", "", regex=False).str.strip()
    literature["Title_Normalised"] = literature["Title"].fillna("").astype(str).str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
    literature["Record_Key"] = literature.apply(lambda r: "doi:" + r["DOI_Normalised"] if r["DOI_Normalised"] else "title:" + r["Title_Normalised"], axis=1)
    literature = literature.drop_duplicates(subset=["CID", "Evidence_Type", "Record_Key"])
    textcol = (literature["Title"].fillna("") + " " + literature["Abstract"].fillna("")).str.lower()
    literature["Breast_Cancer_Term_Match"] = textcol.str.contains(r"breast cancer|breast carcinoma|mcf-7|mda-mb-231|triple-negative", regex=True)
    literature["Cancer_Term_Match"] = textcol.str.contains(r"cancer|carcinoma|tumou?r|anticancer|cytotoxic|neoplasm", regex=True)
    literature["ALDH1A1_Term_Match"] = textcol.str.contains(r"aldh1a1|aldehyde dehydrogenase 1a1", regex=True)
    literature["Exact_Compound_Evidence_Confirmed"] = ""
    literature["Scaffold_Evidence_Confirmed"] = ""
    literature["Experimental_Model"] = ""
    literature["Main_Finding"] = ""
    literature["Reviewer_Notes"] = ""

lit_path = BASE / "05_Literature_Analysis" / "05_ALDH1A1_Literature_Search.xlsx"
with pd.ExcelWriter(lit_path, engine="openpyxl") as w:
    log_df.to_excel(w, sheet_name="Search_Log", index=False)
    literature.to_excel(w, sheet_name="Literature_Records", index=False)
    if not literature.empty:
        literature.drop_duplicates(subset=["Record_Key"]).to_excel(w, sheet_name="Unique_Publications", index=False)
print("Searches attempted:", len(log_df))
print("Successful searches:", int(log_df["Successful"].sum()) if len(log_df) else 0)
print("Retrieved records:", len(literature))
print("ALDH1A1 term matches:", int(literature["ALDH1A1_Term_Match"].sum()) if len(literature) else 0)
print("Breast-cancer term matches:", int(literature["Breast_Cancer_Term_Match"].sum()) if len(literature) else 0)
print("Saved:", lit_path)


Searches attempted: 40
Successful searches: 40
Retrieved records: 202
ALDH1A1 term matches: 0
Breast-cancer term matches: 115
Saved: /content/ALDH1A1_Top10_to_Top1/05_Literature_Analysis/05_ALDH1A1_Literature_Search.xlsx


## 7. PubMed abstract retrieval and evidence classification

In [8]:
# Retrieve PubMed records for unique titles/DOIs from OpenAlex search results.
# This is a complementary database search, not a guarantee of exhaustive coverage.

pubmed_rows, pubmed_errors = [], []
queries = []
if not literature.empty:
    for _, r in literature.drop_duplicates(subset=["Title"]).iterrows():
        title = str(r.get("Title", "")).strip()
        if title and title.lower() != "nan":
            queries.append((str(r.get("CID", "")), title))

for cid, title in queries[:150]:
    try:
        s = session.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                        params={"db":"pubmed","term":f'"{title}"[Title]','retmode':'json','retmax':3}, timeout=25)
        s.raise_for_status()
        ids = s.json().get("esearchresult", {}).get("idlist", [])
        for pmid in ids:
            f = session.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
                            params={"db":"pubmed","id":pmid,"retmode":"xml"}, timeout=25)
            f.raise_for_status()
            from xml.etree import ElementTree as ET
            root = ET.fromstring(f.text)
            title_node = root.find(".//ArticleTitle")
            abstract_nodes = root.findall(".//Abstract/AbstractText")
            abstract = " ".join(" ".join(x.itertext()) for x in abstract_nodes)
            pubmed_rows.append({"CID":cid,"PMID":pmid,"Title": "".join(title_node.itertext()) if title_node is not None else title,
                                "Abstract":abstract,"URL":f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"})
        time.sleep(0.12)
    except Exception as e:
        pubmed_errors.append({"CID":cid,"Title":title,"Error":repr(e)})

pubmed_df = pd.DataFrame(pubmed_rows).drop_duplicates(subset=["PMID"]) if pubmed_rows else pd.DataFrame(columns=["CID","PMID","Title","Abstract","URL"])
if not pubmed_df.empty:
    txt = (pubmed_df["Title"].fillna("") + " " + pubmed_df["Abstract"].fillna("")).str.lower()
    pubmed_df["ALDH1A1_Term_Match"] = txt.str.contains(r"aldh1a1|aldehyde dehydrogenase 1a1", regex=True)
    pubmed_df["Breast_Cancer_Term_Match"] = txt.str.contains(r"breast cancer|breast carcinoma|mcf-7|mda-mb-231|triple-negative", regex=True)
    pubmed_df["Cancer_Term_Match"] = txt.str.contains(r"cancer|carcinoma|tumou?r|anticancer|cytotoxic", regex=True)
    pubmed_df["Exact_Compound_Evidence_Confirmed"] = ""
    pubmed_df["Scaffold_Evidence_Confirmed"] = ""
    pubmed_df["Experimental_Model"] = ""
    pubmed_df["Main_Finding"] = ""
    pubmed_df["Reviewer_Notes"] = ""

pubmed_path = BASE / "05_Literature_Analysis" / "06_ALDH1A1_PubMed_Abstract_Verification.xlsx"
with pd.ExcelWriter(pubmed_path, engine="openpyxl") as w:
    pubmed_df.to_excel(w, sheet_name="PubMed_Abstracts", index=False)
    pd.DataFrame(pubmed_errors).to_excel(w, sheet_name="Retrieval_Errors", index=False)
print("PubMed records retrieved:", len(pubmed_df))
print("PubMed retrieval errors:", len(pubmed_errors))
print("Saved:", pubmed_path)


PubMed records retrieved: 0
PubMed retrieval errors: 87
Saved: /content/ALDH1A1_Top10_to_Top1/05_Literature_Analysis/06_ALDH1A1_PubMed_Abstract_Verification.xlsx


## 8. Integrated candidate comparison and endpoint-level screening

In [9]:
# Merge docking, RDKit, ADMET and literature summaries without creating a weighted composite score.
integrated = raw.merge(descriptors, on="PubChem CID", how="left", validate="one_to_one")
if admet_status == "SUCCESS":
    # Remove duplicate docking fields from ADMET table before merge.
    admet_merge = admet_results.drop(columns=["Overall Rank", "Library", "Edock", "SMILES"], errors="ignore")
    integrated = integrated.merge(admet_merge, on="PubChem CID", how="left", validate="one_to_one")

if not literature.empty:
    lit_summary = literature.groupby("CID").agg(
        Literature_Records=("Title","count"),
        Breast_Cancer_Term_Matches=("Breast_Cancer_Term_Match","sum"),
        Cancer_Term_Matches=("Cancer_Term_Match","sum"),
        ALDH1A1_Term_Matches=("ALDH1A1_Term_Match","sum")
    ).reset_index().rename(columns={"CID":"PubChem CID"})
    integrated = integrated.merge(lit_summary, on="PubChem CID", how="left", validate="one_to_one")
else:
    integrated["Literature_Records"] = 0
    integrated["Breast_Cancer_Term_Matches"] = 0
    integrated["Cancer_Term_Matches"] = 0
    integrated["ALDH1A1_Term_Matches"] = 0

for c in ["Literature_Records","Breast_Cancer_Term_Matches","Cancer_Term_Matches","ALDH1A1_Term_Matches"]:
    integrated[c] = integrated[c].fillna(0).astype(int)

# The six binary-like endpoints use >=0.5 only as a provisional screening flag.
screen_endpoints = [c for c in ["AMES","Carcinogens_Lagunin","ClinTox","DILI","Skin_Reaction","hERG"] if c in integrated.columns]
if screen_endpoints:
    integrated["Classifier_Signals_At_Or_Above_0.5"] = (
        integrated[screen_endpoints].apply(pd.to_numeric, errors="coerce") >= 0.5
    ).sum(axis=1)
else:
    integrated["Classifier_Signals_At_Or_Above_0.5"] = np.nan

integrated["Docking_Score_Interpretation"] = "More negative Edock is a better score within this docking run; not binding affinity."
integrated["ADMET_Interpretation"] = "Model prediction only; not experimental or clinical evidence."
integrated["Literature_Interpretation"] = "Keyword/database retrieval only; manual paper and compound-identity verification required."
integrated["Structure_Interpretation"] = "CID-matched SDF record; exact docked derivative identity must be checked against original ligand source."

# Evidence-informed disposition, deliberately not an automatic composite score or an asserted lead.
def preliminary_disposition(r):
    if r.get("Structure_Identity_Flag","").startswith("CID mismatch"):
        return "HOLD — structure/CID mismatch", "Do not advance until structure identity is resolved."
    if admet_status != "SUCCESS":
        return "PENDING — ADMET unavailable", "Do not assign a final Top-1 until ADMET-AI predictions are available and reviewed."
    if pd.isna(r.get("Classifier_Signals_At_Or_Above_0.5")):
        return "PENDING — endpoint review", "Review available ADMET endpoints individually."
    if int(r.get("Classifier_Signals_At_Or_Above_0.5",0)) >= 4:
        return "LOWER PRIORITY — multiple model flags", "Multiple classifier outputs at or above the exploratory 0.5 threshold; inspect each endpoint."
    if int(r.get("Total_Property_Flags",0)) >= 3:
        return "RESERVE — property liabilities", "Multiple exploratory physicochemical/structural flags; not an automatic exclusion."
    return "RETAIN FOR COMPARATIVE REVIEW", "No automatic exclusion triggered; compare docking, endpoint-level ADMET, structure identity and literature manually."

disp = integrated.apply(preliminary_disposition, axis=1, result_type="expand")
integrated["Preliminary_Disposition"] = disp[0]
integrated["Disposition_Rationale"] = disp[1]
integrated["Manual_Review_Status"] = "PENDING"
integrated["Final_Top1_Selection"] = "NOT SELECTED — manual evidence review required"

integrated = integrated.sort_values("Overall Rank")
out = BASE / "06_Integrated_Candidate_Assessment" / "07_ALDH1A1_Integrated_Top10_Comparison.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as w:
    integrated.to_excel(w, sheet_name="Integrated_Top10", index=False)
    integrated[["PubChem CID","Overall Rank","Edock_Rank_Recalculated","Edock","Total_Property_Flags","Classifier_Signals_At_Or_Above_0.5","Literature_Records","Preliminary_Disposition","Disposition_Rationale","Manual_Review_Status","Final_Top1_Selection"]].to_excel(w, sheet_name="Candidate_Screening", index=False)
    pd.DataFrame({"Endpoint":screen_endpoints,"Threshold":[0.5]*len(screen_endpoints),"Interpretation":["Exploratory flag only; not a validated clinical cutoff"]*len(screen_endpoints)}).to_excel(w, sheet_name="Classifier_Thresholds", index=False)

print("ADMET status:", admet_status)
print("Integrated candidates:", len(integrated))
display(integrated[["Overall Rank","PubChem CID","Library","Edock","Edock_Rank_Recalculated","Total_Property_Flags","Classifier_Signals_At_Or_Above_0.5","Literature_Records","Preliminary_Disposition"]])
print("Saved:", out)


ADMET status: SUCCESS
Integrated candidates: 10


,Overall Rank,PubChem CID,Library,Edock,Edock_Rank_Recalculated,Total_Property_Flags,Classifier_Signals_At_Or_Above_0.5,Literature_Records,Preliminary_Disposition
0,1,173315143,Thiomorpholine,-27.35929,10,1,1,20,RETAIN FOR COMPARATIVE REVIEW
1,2,153531369,Thiophene,-200.09910,1,5,2,20,RESERVE — property liabilities
2,3,150310321,Thiophene,-130.70490,3,4,2,20,RESERVE — property liabilities
3,4,153531370,Thiophene,-153.06790,2,4,2,20,RESERVE — property liabilities
4,5,175018237,Thiophene,-129.09780,4,4,2,20,RESERVE — property liabilities
5,6,12592795,Indole,-104.91610,5,4,2,20,RESERVE — property liabilities
6,7,12592794,Indole,-88.92295,6,4,3,20,RESERVE — property liabilities
7,8,21260039,Benzoxazole,-86.31135,8,4,2,20,RESERVE — property liabilities
8,9,58965179,Benzimidazole,-88.55415,7,0,2,22,RETAIN FOR COMPARATIVE REVIEW
9,10,14157051,Benzimidazole,-78.31743,9,4,2,20,RESERVE — property liabilities


Saved: /content/ALDH1A1_Top10_to_Top1/06_Integrated_Candidate_Assessment/07_ALDH1A1_Integrated_Top10_Comparison.xlsx


## 9. Endpoint inventory and toxicity summary

In [10]:
endpoint_inventory = pd.DataFrame()
tox_summary = pd.DataFrame()
if admet_status == "SUCCESS":
    endpoint_cols = [c for c in admet_results.columns if c not in ["PubChem CID","Overall Rank","Library","Edock","SMILES"]]
    inv = []
    for c in endpoint_cols:
        vals = pd.to_numeric(admet_results[c], errors="coerce")
        inv.append({"Endpoint":c,"Numeric_Values":int(vals.notna().sum()),"Missing":int(vals.isna().sum()),"Minimum":vals.min(),"Maximum":vals.max(),"Mean":vals.mean()})
    endpoint_inventory = pd.DataFrame(inv)
    tox_cols = [c for c in endpoint_cols if c in ["AMES","Carcinogens_Lagunin","ClinTox","DILI","hERG","Skin_Reaction","NR-AR","NR-AR-LBD","NR-AhR","NR-Aromatase","NR-ER","NR-ER-LBD","NR-PPAR-gamma","SR-ARE","SR-ATAD5","SR-HSE","SR-MMP","SR-p53","LD50_Zhu"]]
    tox_summary = admet_results[["PubChem CID"] + tox_cols].copy()
    tox_path = BASE / "04_Molecular_Properties_and_ADMET" / "08_ALDH1A1_ADMET_Endpoint_Audit.xlsx"
    with pd.ExcelWriter(tox_path, engine="openpyxl") as w:
        endpoint_inventory.to_excel(w, sheet_name="Endpoint_Inventory", index=False)
        tox_summary.to_excel(w, sheet_name="Toxicity_Endpoints", index=False)
    print("Endpoint inventory:", len(endpoint_inventory))
    print("Toxicity endpoints:", len(tox_cols))
    print("Saved:", tox_path)
else:
    print("ADMET endpoint audit skipped because ADMET-AI did not complete.")


Endpoint inventory: 105
Toxicity endpoints: 19
Saved: /content/ALDH1A1_Top10_to_Top1/04_Molecular_Properties_and_ADMET/08_ALDH1A1_ADMET_Endpoint_Audit.xlsx


## 10. Final candidate decision matrix (no fabricated Top-1)

In [11]:
decision_cols = [
    "PubChem CID","Library","Overall Rank","Edock_Rank_Recalculated",
    "Original Edock","Edock","Redocking Edock",
    "Molecular_Weight_RDKit","cLogP_RDKit","TPSA","HBD","HBA","QED",
    "Lipinski_Violations","PAINS_Alert","Total_Property_Flags",
    "Classifier_Signals_At_Or_Above_0.5",
    "Literature_Records","Breast_Cancer_Term_Matches","Cancer_Term_Matches","ALDH1A1_Term_Matches",
    "Preliminary_Disposition","Disposition_Rationale",
    "Manual_Review_Status","Final_Top1_Selection"
]
decision = integrated[[c for c in decision_cols if c in integrated.columns]].copy()
decision["Reviewer_Exact_Compound_Evidence"] = ""
decision["Reviewer_Scaffold_Evidence"] = ""
decision["Reviewer_ADMET_Interpretation"] = ""
decision["Reviewer_Final_Decision"] = ""
decision["Reviewer_Rationale"] = ""
decision["Evidence_Limitations"] = (
    "No experimental ALDH1A1 inhibition or cellular validation is established by this computational workflow; "
    "literature search matches require manual verification; docking score is not binding affinity."
)
decision_path = BASE / "06_Integrated_Candidate_Assessment" / "09_ALDH1A1_Final_Decision_Matrix.xlsx"
with pd.ExcelWriter(decision_path, engine="openpyxl") as w:
    decision.to_excel(w, sheet_name="Final_Decision_Matrix", index=False)
    pd.DataFrame({
        "Assessment":["Candidates evaluated","Valid SDF records","CID matches","ADMET-AI status","Final Top-1 selected"],
        "Result":[len(raw),len(sdf_df),int(match["CID_Match"].sum()),admet_status,"No — manual evidence review required"]
    }).to_excel(w, sheet_name="Evidence_Summary", index=False)
print("Decision matrix saved:", decision_path)
display(decision[["PubChem CID","Overall Rank","Edock_Rank_Recalculated","Preliminary_Disposition","Final_Top1_Selection"]])


Decision matrix saved: /content/ALDH1A1_Top10_to_Top1/06_Integrated_Candidate_Assessment/09_ALDH1A1_Final_Decision_Matrix.xlsx


,PubChem CID,Overall Rank,Edock_Rank_Recalculated,Preliminary_Disposition,Final_Top1_Selection
0,173315143,1,10,RETAIN FOR COMPARATIVE REVIEW,NOT SELECTED — manual evidence review required
1,153531369,2,1,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
2,150310321,3,3,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
3,153531370,4,2,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
4,175018237,5,4,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
5,12592795,6,5,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
6,12592794,7,6,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
7,21260039,8,8,RESERVE — property liabilities,NOT SELECTED — manual evidence review required
8,58965179,9,7,RETAIN FOR COMPARATIVE REVIEW,NOT SELECTED — manual evidence review required
9,14157051,10,9,RESERVE — property liabilities,NOT SELECTED — manual evidence review required


## 11. Selected-candidate SDF export — run after manual Top-1 decision

In [12]:
# Enter the CID chosen after reviewing the decision matrix and evidence.
# Leave as None until a Top-1 has actually been selected.
SELECTED_CID = None  # Example format: "58965179"

if SELECTED_CID is None:
    print("No candidate exported: SELECTED_CID is None.")
    print("This prevents an unsupported or fabricated Top-1 selection.")
else:
    SELECTED_CID = clean_cid(SELECTED_CID)
    hits = sdf_df[sdf_df["SDF_CID"] == SELECTED_CID]
    if len(hits) != 1:
        raise ValueError("Selected CID does not map to exactly one validated SDF record.")
    mols = Chem.SDMolSupplier(str(SDF), removeHs=False)
    rec = int(hits.iloc[0]["SDF_Record"])
    selected = mols[rec-1]
    if selected is None:
        raise ValueError("Selected SDF record cannot be parsed.")
    selected.SetProp("PUBCHEM_CID", SELECTED_CID)
    selected.SetProp("SELECTION_STATUS", "User-reviewed Top-1 candidate")
    selected.SetProp("TARGET", "ALDH1A1")
    selected.SetProp("TARGET_PDB", "4X4L")
    out_sdf = BASE / "09_Selected_Candidate_Structure" / f"01_ALDH1A1_Selected_Top1_CID_{SELECTED_CID}.sdf"
    writer = Chem.SDWriter(str(out_sdf))
    writer.write(selected)
    writer.close()
    check = [m for m in Chem.SDMolSupplier(str(out_sdf), removeHs=False) if m is not None]
    if len(check) != 1:
        raise RuntimeError("Selected SDF export verification failed.")
    print("Selected SDF verified:", out_sdf)


No candidate exported: SELECTED_CID is None.
This prevents an unsupported or fabricated Top-1 selection.


## 12. Supervisor-ready report and computational methodology

In [13]:
from docx import Document
from docx.shared import Pt

REPORT_DIR = BASE / "07_Final_Reports"
METHOD_DIR = BASE / "08_Methodology"

doc = Document()
doc.styles["Normal"].font.name = "Arial"
doc.styles["Normal"].font.size = Pt(10)
doc.add_heading("ALDH1A1 Top-10 Candidate Evaluation", 0)
doc.add_paragraph("Target structure: ALDH1A1 | PDB: 4X4L")
doc.add_paragraph("Evaluation status: computational prioritisation package; final Top-1 awaits manual evidence review.")

doc.add_heading("1. Input and docking audit", level=1)
doc.add_paragraph(f"Ten candidates were evaluated from the supplied authoritative workbook. The workbook's Overall Rank was preserved, and Edock was independently ranked in ascending order. The original and redocking scores were retained separately for discrepancy review.")
doc.add_paragraph(f"Workbook: {XLSX.name}. Ligand structure source: {SDF.name}.")
doc.add_heading("2. Structure validation", level=1)
doc.add_paragraph(f"The SDF contained {len(sdf_df)} valid records and {len(invalid_records)} invalid records. CID matching identified {int(match['CID_Match'].sum())} matching candidates. CID matching does not by itself prove that the PubChem structure is identical to the exact docked derivative; the source workbook's structure warnings are retained.")
doc.add_heading("3. Molecular properties", level=1)
doc.add_paragraph("RDKit descriptors include molecular weight, exact molecular weight, calculated logP, TPSA, HBD, HBA, rotatable bonds, ring counts, fraction Csp3, heavy atoms, formal charge, QED, Lipinski violations and PAINS alerts. Property flags are exploratory and are not automatic drug-candidate exclusions.")
doc.add_heading("4. ADMET-AI", level=1)
doc.add_paragraph(f"ADMET-AI status: {admet_status}. Predictions, if available, are computational model outputs and not experimental or clinical evidence. The ≥0.5 classifier threshold is an exploratory screening convention only. DrugBank-approved percentiles, if present, are relative reference-distribution values.")
if admet_error:
    doc.add_paragraph("ADMET-AI error/log: " + admet_error)
doc.add_heading("5. Literature evidence", level=1)
doc.add_paragraph(f"OpenAlex literature records retrieved: {len(literature)}. PubMed abstract records retrieved: {len(pubmed_df)}. Keyword matches are retrieval/screening signals only. Exact-compound activity, scaffold activity and ALDH1A1-target relevance require manual paper-level verification.")
doc.add_heading("6. Integrated assessment and candidate disposition", level=1)
doc.add_paragraph("Docking, molecular properties, endpoint-level ADMET predictions, structure identity and literature evidence are presented separately. No arbitrary weighted composite score was created. No Top-1 candidate is declared automatically; the decision matrix includes manual-review fields to prevent a fabricated or unsupported lead selection.")
doc.add_heading("7. Limitations", level=1)
for item in [
    "Docking scores are not experimental binding affinities.",
    "A PubChem CID match does not independently establish exact docked-derivative identity.",
    "ADMET-AI outputs are predictions and require experimental follow-up.",
    "Literature search results are not exhaustive and keyword matches require manual verification.",
    "No experimental ALDH1A1 enzyme inhibition, selectivity, cellular or in-vivo validation is established by this workflow."
]:
    doc.add_paragraph(item, style="List Bullet")
doc.save(REPORT_DIR / "01_ALDH1A1_Supervisor_Ready_Evaluation_Report.docx")

method = Document()
method.styles["Normal"].font.name = "Arial"
method.styles["Normal"].font.size = Pt(10)
method.add_heading("ALDH1A1 Top-10 to Top-1 Computational Methodology", 0)
method.add_paragraph("Target: ALDH1A1 | PDB: 4X4L")
sections = {
"1. Source files":[f"Authoritative docking workbook: {XLSX.name}.",f"Exact ligand SDF: {SDF.name}.","Both source files were copied unchanged into the project source folder."],
"2. Docking ranking":["The workbook Overall Rank was preserved as supplied.","Edock was independently ranked ascending; more negative values indicate better docking scores within the reported run.","Original Edock and redocking Edock were retained separately and score differences were calculated."],
"3. Structure validation":["RDKit parsed and sanitised each SDF record.","SDF records were matched to the workbook using PubChem CID.","Molecular formula, molecular weight, canonical isomeric SMILES, InChIKey, heavy-atom count and isotope count were recorded.","CID match is not considered proof of exact docked-derivative identity."],
"4. Molecular descriptors":["RDKit calculated MW, exact MW, cLogP, TPSA, HBD, HBA, rotatable bonds, ring counts, fraction Csp3, heavy atoms, formal charge and QED.","Lipinski violations and PAINS alerts were recorded.","Property flags are exploratory indicators, not automatic exclusion criteria."],
"5. ADMET-AI":["ADMET-AI was applied to the SDF-derived canonical isomeric SMILES where service/package execution succeeded.","All returned endpoints were retained; toxicity-related endpoints and endpoint inventory were separated.","Classifier values ≥0.5 were counted as exploratory flags only.","Percentile outputs are relative to the DrugBank-approved reference distribution, not probabilities."],
"6. Literature retrieval":["OpenAlex searches covered exact CIDs, compound names, scaffold anticancer and scaffold breast-cancer terms.","PubMed title/abstract retrieval was attempted for returned publication titles.","Search logs, records, abstracts and reviewer fields were retained.","Keyword matches are not confirmed evidence of exact-compound activity."],
"7. Integrated assessment":["Docking, properties, ADMET, structure identity and literature evidence were considered separately.","No arbitrary composite score was calculated.","Final Top-1 selection requires manual review of the evidence and completion of reviewer fields."],
"8. Limitations":["No experimental ALDH1A1 inhibition or cellular validation is implied.","Docking scores are not binding affinities.","Predictions and literature search results require independent verification."]
}
for h, items in sections.items():
    method.add_heading(h, level=1)
    for item in items: method.add_paragraph(item, style="List Bullet")
method.add_heading("9. Reproducibility", level=1)
method.add_paragraph("The notebook saves intermediate workbooks, literature logs, integrated comparisons, reports and the final ZIP archive under /content.")
method.save(METHOD_DIR / "01_ALDH1A1_Computational_Methodology_and_Limitations.docx")

print("Reports created.")
print("Supervisor report:", REPORT_DIR / "01_ALDH1A1_Supervisor_Ready_Evaluation_Report.docx")
print("Methodology:", METHOD_DIR / "01_ALDH1A1_Computational_Methodology_and_Limitations.docx")


Reports created.
Supervisor report: /content/ALDH1A1_Top10_to_Top1/07_Final_Reports/01_ALDH1A1_Supervisor_Ready_Evaluation_Report.docx
Methodology: /content/ALDH1A1_Top10_to_Top1/08_Methodology/01_ALDH1A1_Computational_Methodology_and_Limitations.docx


## 13. Final package audit, ZIP creation and download

In [14]:
# Copy all generated intermediate outputs into the nine-folder deliverable structure.
# All files are already being written beneath BASE.

# Create a project manifest with file sizes and SHA-256 hashes.
manifest = []
for p in sorted(BASE.rglob("*")):
    if p.is_file():
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest.append({
            "Relative_Path": str(p.relative_to(BASE)),
            "Size_Bytes": p.stat().st_size,
            "SHA256": digest
        })
manifest_df = pd.DataFrame(manifest)
manifest_path = BASE / "07_Final_Reports" / "02_ALDH1A1_Project_File_Manifest.xlsx"
manifest_df.to_excel(manifest_path, index=False)

# Add run summary.
summary = {
    "Target": "ALDH1A1",
    "Target_PDB": "4X4L",
    "Candidates": int(len(raw)),
    "Valid_SDF_Records": int(len(sdf_df)),
    "CID_Matches": int(match["CID_Match"].sum()),
    "ADMET_AI_Status": admet_status,
    "OpenAlex_Literature_Records": int(len(literature)),
    "PubMed_Abstract_Records": int(len(pubmed_df)),
    "Final_Top1_Selected": "No — manual review required",
    "Important_Limitation": "CID match does not alone prove exact docked-derivative identity."
}
pd.DataFrame([summary]).to_excel(BASE / "07_Final_Reports" / "03_ALDH1A1_Run_Summary.xlsx", index=False)

# Build one ZIP containing the complete project folder.
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in BASE.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(Path(BASE.name) / p.relative_to(BASE)))

# Verify ZIP integrity and required deliverables.
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad = z.testzip()
    names = z.namelist()

required_files = [
    "01_Source_Files/01_Authoritative_ALDH1A1_Top10_Docking_Input.xlsx",
    "01_Source_Files/02_ALDH1A1_Top10_Exact_Ligand_Structures.sdf",
    "02_Data_Quality_and_Structure_Audit/01_ALDH1A1_Compound_Data_Audit.xlsx",
    "02_Data_Quality_and_Structure_Audit/02_ALDH1A1_SDF_Structure_and_CID_Matching.xlsx",
    "04_Molecular_Properties_and_ADMET/03_ALDH1A1_RDKit_Molecular_Properties.xlsx",
    "06_Integrated_Candidate_Assessment/07_ALDH1A1_Integrated_Top10_Comparison.xlsx",
    "06_Integrated_Candidate_Assessment/09_ALDH1A1_Final_Decision_Matrix.xlsx",
    "07_Final_Reports/01_ALDH1A1_Supervisor_Ready_Evaluation_Report.docx",
    "08_Methodology/01_ALDH1A1_Computational_Methodology_and_Limitations.docx"
]
missing_required = [f for f in required_files if str(Path(BASE.name) / f) not in names]
if bad is not None:
    raise RuntimeError(f"ZIP integrity test failed at: {bad}")
if missing_required:
    raise RuntimeError(f"Required package files missing: {missing_required}")

print("FINAL PACKAGE AUDIT")
print("===================")
print("Project folder:", BASE)
print("Files packaged:", len(names))
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))
print("ZIP integrity: PASS")
print("Required files: PASS")
print("ADMET-AI status:", admet_status)
print("Top-1 selected:", summary["Final_Top1_Selected"])
print("\\nFolder structure:")
for folder in FOLDER_NAMES:
    count = sum(1 for p in (BASE / folder).rglob("*") if p.is_file())
    print(folder, "|", count, "files")

files.download(str(ZIP_PATH))


FINAL PACKAGE AUDIT
Project folder: /content/ALDH1A1_Top10_to_Top1
Files packaged: 15
ZIP size (MB): 0.44
ZIP integrity: PASS
Required files: PASS
ADMET-AI status: SUCCESS
Top-1 selected: No — manual review required
\nFolder structure:
01_Source_Files | 2 files
02_Data_Quality_and_Structure_Audit | 2 files
03_Docking_Results | 0 files
04_Molecular_Properties_and_ADMET | 3 files
05_Literature_Analysis | 2 files
06_Integrated_Candidate_Assessment | 2 files
07_Final_Reports | 3 files
08_Methodology | 1 files
09_Selected_Candidate_Structure | 0 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ============================================================
# ALDH1A1 — EXPORT CANDIDATE SDF FILES SEPARATELY
# Target PDB: 4X4L
# ============================================================

from pathlib import Path
from rdkit import Chem
from google.colab import files

# Automatically locate original SDF
sdf_matches = list(Path("/content").rglob(
    "ALDH1A1_4X4L_Overall_Top10_Exact.sdf"
))

if not sdf_matches:
    sdf_matches = list(Path("/content").rglob("*ALDH1A1*Top10*.sdf"))

if not sdf_matches:
    raise FileNotFoundError("Original ALDH1A1 Top-10 SDF not found. Upload it.")

INPUT_SDF = sdf_matches[0]

# Two candidates — different ranking bases
CANDIDATES = {
    "CID_153531369_Best_Edock_Provisional": {
        "cid": "153531369",
        "status": "Best recalculated Edock; provisional, not validated Top-1"
    },
    "CID_173315143_Original_Rank1_Provisional": {
        "cid": "173315143",
        "status": "Original workbook Overall Rank 1; provisional, not validated Top-1"
    }
}

supplier = Chem.SDMolSupplier(str(INPUT_SDF), removeHs=False)

def get_cid(mol):
    keys = [
        "PUBCHEM_COMPOUND_CID",
        "PUBCHEM_CID",
        "PubChem_CID",
        "PubChem CID"
    ]
    for key in keys:
        if mol.HasProp(key):
            return mol.GetProp(key).strip()

    # Fallback: inspect all properties
    for key in mol.GetPropNames(includePrivate=True, includeComputed=False):
        if "cid" in key.lower():
            value = mol.GetProp(key).strip()
            if value.isdigit():
                return value

    return ""

valid_molecules = [m for m in supplier if m is not None]

print("Valid SDF records:", len(valid_molecules))

for filename, info in CANDIDATES.items():

    cid = info["cid"]

    matches = [
        mol for mol in valid_molecules
        if get_cid(mol) == cid
    ]

    if len(matches) != 1:
        raise ValueError(
            f"CID {cid}: expected exactly one SDF record; found {len(matches)}"
        )

    mol = Chem.Mol(matches[0])

    mol.SetProp("PUBCHEM_CID", cid)
    mol.SetProp("TARGET", "ALDH1A1")
    mol.SetProp("TARGET_PDB", "4X4L")
    mol.SetProp("CANDIDATE_STATUS", info["status"])

    output = Path("/content") / f"ALDH1A1_{filename}.sdf"

    writer = Chem.SDWriter(str(output))
    writer.write(mol)
    writer.close()

    # Verify exported SDF
    check = [m for m in Chem.SDMolSupplier(str(output), removeHs=False)
             if m is not None]

    if len(check) != 1 or get_cid(check[0]) != cid:
        raise RuntimeError(f"Export verification failed for CID {cid}")

    print("VERIFIED:", output.name)

    files.download(str(output))

print("\nBOTH CANDIDATE SDF FILES EXPORTED AND VERIFIED.")

Valid SDF records: 10
VERIFIED: ALDH1A1_CID_153531369_Best_Edock_Provisional.sdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

VERIFIED: ALDH1A1_CID_173315143_Original_Rank1_Provisional.sdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


BOTH CANDIDATE SDF FILES EXPORTED AND VERIFIED.
